# Agent traversal and query implementation

Loads finished tree. Agent walks tree w/o vector similarity including backtracking and working memory. Virtual subfolders acreated to minimize branching factor to <8. These virtual subfolders are created by embedding the child summaries and k-means-ing them into semantic groups, so similar files sit together.*Embeddings are used only to organise the groups, so navigation is still agent-driven

In [15]:
%pip install ollama numpy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, json, re, time, hashlib, math, textwrap
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional

import ollama
import numpy as np

# models
OLLAMA_URL  = "http://localhost:11528"
AGENT_MODEL = "gpt-oss:120b"        # navigation decisions + final answer (text-only is fine)
EMBED_MODEL = "nomic-embed-text"    # ONLY for organising virtual subfolders (cluster strategy)

# paths
CACHE_DIR = Path("tree_cache")
TREE_FILE = CACHE_DIR / "corpus_tree.json"

MAX_BRANCH        = 8        # max branching factor
VIRTUAL_TARGET    = 6        # cluster strategy aims for ~this many items per group
VIRTUAL_STRATEGY  = "cluster"  # cluster, chunk or none (cluster is most logical)
VIRTUAL_SUMMARY_LLM = True   # summarise each virtual group with the LLM as if it was its own folder

# agent settings
MAX_STEPS        = 40        # safety cap on traversal steps per query
MEMORY_MAX_ITEMS = 20        # working-memory notes kept
MAX_EVIDENCE     = 12        # leaf/nodes fed into the final answer

GEN_NUM_PREDICT  = 1000000   # effectively uncapped so answers finish
KEEP_ALIVE       = "30m"
DISABLE_THINKING = False     # no gpt-oss reasoning
RETRY_WAIT_MAX   = 60        # wait-and-retry if Ollama disconnects
READY_POLL       = 10

print(f"Agent model: {AGENT_MODEL} | embed (cluster only): {EMBED_MODEL}")
print(f"Tree file  : {TREE_FILE}")
print(f"Strategy   : {VIRTUAL_STRATEGY}  (max_branch={MAX_BRANCH})")

Agent model: gpt-oss:120b | embed (cluster only): nomic-embed-text
Tree file  : tree_cache/corpus_tree.json
Strategy   : cluster  (max_branch=8)


In [ ]:
OLLAMA_TIMEOUT = 600  # per-request timeout in seconds; generous since the agent model can be slow to answer
client = ollama.Client(host=OLLAMA_URL, timeout=OLLAMA_TIMEOUT)  # the shared client every call goes through, pointed at the tunnel
_THINK = {"use": not DISABLE_THINKING}  # mutable flag for whether to send think=true; note it's flipped from the build notebooks, here reasoning is ON by default because it helps navigation


def _model_names(r) -> List[str]:
    # pulls the plain list of model name strings out of client.list(), coping with whichever response shape the ollama version hands back
    raw = r.get("models", []) if hasattr(r, "get") else getattr(r, "models", [])
    out = []
    for m in raw:
        n = getattr(m, "model", None) or getattr(m, "name", None)
        if n is None and isinstance(m, dict):
            n = m.get("model") or m.get("name")
        if n:
            out.append(n)
    return out


def _wait_backoff(attempt):
    # sleeps a bit longer each retry, 5s 10s 20s 40s 80s but capped, so we don't pound the server while it's down
    time.sleep(min(RETRY_WAIT_MAX, 5 * (2 ** min(attempt - 1, 4))))


def _wait_until_ready():
    # waits for the server and the models it needs before anything runs, only needing the embed model when the cluster strategy is on, and loops forever rather than crashing if things aren't up yet
    need = [AGENT_MODEL] + ([EMBED_MODEL] if VIRTUAL_STRATEGY == "cluster" else [])
    announced = False
    while True:
        try:
            names = _model_names(client.list())
            missing = [m for m in need if not any(m in n for n in names)]
            if not missing:
                if announced:
                    print("Ollama reconnected.")
                print(f"Ollama OK — models present: {', '.join(need)}")
                return
            reason = f"model(s) not loaded yet: {missing}"
        except Exception as e:
            reason = f"server unreachable ({type(e).__name__}: {e})"
        if not announced:
            print(f"Waiting for Ollama — {reason}. Re-checking every {READY_POLL}s; won't stop.")
            announced = True
        time.sleep(READY_POLL)


def _chat_text(prompt: str) -> str:
    # the agent's one text call, takes a prompt and returns the reply string, and like the build notebooks it waits and retries forever if Ollama drops instead of dying mid-query
    opts = {"temperature": 0, "num_predict": GEN_NUM_PREDICT}
    attempt = 0
    while True:
        kw = dict(model=AGENT_MODEL, messages=[{"role": "user", "content": prompt}],
                  options=opts, keep_alive=KEEP_ALIVE)
        if _THINK["use"]:
            kw["think"] = True
        try:
            return client.chat(**kw)["message"]["content"].strip()
        except TypeError:
            _THINK["use"] = False
        except Exception as e:
            if _THINK["use"]:
                _THINK["use"] = False
                continue
            attempt += 1
            if attempt == 1 or attempt % 5 == 0:
                print(f"[waiting for Ollama] {type(e).__name__}: {e} — retrying...")
            _wait_backoff(attempt)


def _combine_text(summaries: List[str], label: str) -> str:
    # writes a short blurb for a virtual group from its members' summaries, just enough for the agent to judge whether that group is worth exploring
    joined = "\n".join(f"- {s}" for s in summaries)
    prompt = (
        f"In 2-4 sentences, describe what this group called '{label}' covers, so an "
        "agent can decide whether to explore it. List the main topics/items. "
        f"Respond with ONLY the description.\n\n{joined}")
    return _chat_text(prompt)


def _embed(text: str):
    # gets an embedding vector for text, used only to cluster children into virtual subfolders, and returns None on failure since clustering can fall back to plain chunking
    try:
        return client.embeddings(model=EMBED_MODEL, prompt=text or " ")["embedding"]
    except Exception:
        return None


_wait_until_ready()  # block here at cell run until the server and models are reachable
print("LLM helpers ready (resilient chat + embeddings for clustering)")

Ollama OK — models present: gpt-oss:120b, nomic-embed-text
LLM helpers ready (resilient chat + embeddings for clustering)


In [ ]:
import math  # for ceil, used to work out how many groups to split children into


@dataclass  # auto-generate the boilerplate from the fields below
class TreeNode:  # the node type, a trimmed-down version of the build-notebook one since here we only ever read the tree
    node_id:   str  # unique id; also used as the embed and nav cache key
    node_type: str  # root, folder, document, section, chunk, or the virtual "vgroup" we add here
    name:      str  # human-readable label shown to the agent
    path:      str  # filesystem path where relevant; empty otherwise
    summary:   str  # the summary the agent reads to make decisions
    content:   str = ""  # raw text on leaf chunks; empty on everything else
    children:  List["TreeNode"] = field(default_factory=list)  # child nodes; fresh list per instance
    metadata:  Dict[str, Any]   = field(default_factory=dict)  # extra info like kind, page, source_file; fresh dict per instance

    @classmethod  # builds a node without needing an existing one
    def from_dict(cls, d: Dict) -> "TreeNode":  # rebuilds a node and its whole subtree from the saved json, tolerating missing optional fields
        node = cls(node_id=d["node_id"], node_type=d["node_type"], name=d["name"],
                   path=d.get("path", ""), summary=d.get("summary", ""),
                   content=d.get("content", ""), metadata=d.get("metadata", {}))
        node.children = [cls.from_dict(c) for c in d.get("children", [])]
        return node

    def is_leaf(self) -> bool:  # true only for leaf chunks, which is how the agent knows it's hit the bottom
        return self.node_type == "chunk"

    def count_leaves(self) -> int:  # counts leaf units under this node, recursing through children
        return 1 if self.is_leaf() else sum(c.count_leaves() for c in self.children)


def _vid(parent_id: str, tag: str) -> str:
    # makes a stable id for a virtual group from its parent's id plus a tag, prefixed with v so it never collides with a real node id
    return "v" + hashlib.md5(f"{parent_id}|{tag}".encode()).hexdigest()[:11]


def _clip(text: str, n: int) -> str:
    # tidies whitespace and trims text to n characters with an ellipsis, used to keep summaries short in prompts
    t = re.sub(r"\s+", " ", text or "").strip()
    return t if len(t) <= n else t[:n] + " …"


# embedding/k-means for grouping
_embed_cache: Dict[str, Any] = {}  # caches embeddings by node id so we never embed the same node twice


def _embed_node(node: "TreeNode"):
    # gets the embedding for a node, name plus summary, caching it, used only to cluster children into virtual subfolders, never to answer queries
    if node.node_id in _embed_cache:
        return _embed_cache[node.node_id]
    vec = _embed((node.name + ". " + (node.summary or ""))[:2000])
    _embed_cache[node.node_id] = vec
    return vec


def _kmeans(vectors, k: int, iters: int = 25, seed: int = 0):
    # a tiny self-contained k-means in numpy, it normalises the vectors to unit length so distance behaves like cosine similarity, then iterates assign-and-recentre until the labels stop changing, and is seeded so the grouping is reproducible
    X = np.asarray(vectors, dtype=float)
    n = len(X)
    k = max(1, min(k, n))
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / np.clip(norms, 1e-9, None)          # cosine-ish via unit vectors
    rng = np.random.default_rng(seed)
    C = X[rng.choice(n, size=k, replace=False)].copy()
    labels = np.full(n, -1)
    for _ in range(iters):
        d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1)
        new = d.argmin(1)
        if np.array_equal(new, labels):
            break
        labels = new
        for j in range(k):
            pts = X[labels == j]
            C[j] = pts.mean(0) if len(pts) else X[rng.integers(n)]
    return labels.tolist()


# group construction for virtual subfolders
def _group_summary(members: List["TreeNode"], label: str) -> str:
    # writes the summary that represents a virtual group, asking the model to blurb it if that's switched on, and otherwise falling back to a cheap heuristic listing of the members
    if VIRTUAL_SUMMARY_LLM:
        try:
            return _combine_text([m.summary for m in members], label=label)
        except Exception:
            pass
    lines = [f"- {m.name}: {_clip(m.summary, 160)}" for m in members[:MAX_BRANCH]]
    more = f"\n- …and {len(members) - MAX_BRANCH} more" if len(members) > MAX_BRANCH else ""
    return f"A group of {len(members)} related items:\n" + "\n".join(lines) + more


def _make_vgroup(parent: "TreeNode", members: List["TreeNode"], idx: int, strategy: str) -> "TreeNode":
    # wraps a subset of children in a virtual group node, a fake folder the agent can descend into, carrying its own generated summary and the members as its children
    vid = _vid(parent.node_id, f"{strategy}:{idx}")
    return TreeNode(node_id=vid, node_type="vgroup",
                    name=f"[group {idx + 1} · {len(members)} items]", path="",
                    summary=_group_summary(members, label=f"{parent.name} group {idx + 1}"),
                    children=list(members),
                    metadata={"virtual": True, "strategy": strategy, "size": len(members)})


def _chunk_groups(parent: "TreeNode", kids: List["TreeNode"], strategy: str = "chunk"):
    # the simple grouping strategy, splits children into at most MAX_BRANCH even contiguous groups, no embeddings involved
    n = len(kids)
    size = math.ceil(n / MAX_BRANCH)                 # -> at most MAX_BRANCH groups
    groups = [kids[i:i + size] for i in range(0, n, size)]
    return [_make_vgroup(parent, g, i, strategy) for i, g in enumerate(groups)]


def _cluster_groups(parent: "TreeNode", kids: List["TreeNode"]):
    # the semantic grouping strategy, embeds the children and k-means them so similar files sit together, but it quietly falls back to plain chunking if embeddings aren't available or if clustering fails to actually split anything
    vecs = [_embed_node(k) for k in kids]
    if any(v is None for v in vecs):                 # no embeddings -> fall back
        return _chunk_groups(parent, kids, strategy="cluster-fallback")
    k = max(2, min(MAX_BRANCH, math.ceil(len(kids) / VIRTUAL_TARGET)))
    labels = _kmeans(vecs, k)
    buckets: Dict[int, list] = {}
    for kid, lab in zip(kids, labels):
        buckets.setdefault(lab, []).append(kid)
    # guard against a degenerate "everything in one cluster" -> no progress
    if len(buckets) < 2 or max(len(v) for v in buckets.values()) == len(kids):
        return _chunk_groups(parent, kids, strategy="cluster-fallback")
    ordered = [buckets[lab] for lab in sorted(buckets)]
    return [_make_vgroup(parent, mem, i, "cluster") for i, mem in enumerate(ordered)]


_nav_cache: Dict[tuple, List["TreeNode"]] = {}  # caches the nav children per node and strategy, so a node is only virtualised once


def get_nav_children(node: "TreeNode", strategy: str) -> List["TreeNode"]:
    """Children the agent chooses among at `node` — virtualised to <= MAX_BRANCH."""
    # the one entry point the agent uses, hands back the children to choose among, passing them through untouched when there are few enough or the strategy is none, and otherwise grouping them via chunk or cluster, all cached
    key = (node.node_id, strategy)
    if key in _nav_cache:
        return _nav_cache[key]
    kids = node.children
    if strategy == "none" or len(kids) <= MAX_BRANCH:
        res = list(kids)
    elif strategy == "chunk":
        res = _chunk_groups(node, kids)
    elif strategy == "cluster":
        res = _cluster_groups(node, kids)
    else:
        res = list(kids)
    _nav_cache[key] = res
    return res


print("Virtual-subfolder layer ready (strategies: none | chunk | cluster)")

Virtual-subfolder layer ready (strategies: none | chunk | cluster)


In [ ]:
if not TREE_FILE.exists():
    # bail out early with a clear message if the tree file isn't there, since the agent has nothing to walk without it
    raise SystemExit(f"Tree not found: {TREE_FILE.resolve()} — run the build (prototype 5) first.")

with open(TREE_FILE, encoding="utf-8") as f:
    ROOT = TreeNode.from_dict(json.load(f))  # load the saved json and rebuild the whole tree into ROOT, the node the agent starts from

def _count(n):
    # counts every node in a subtree including itself, recursing through children, just for the stats line below
    return 1 + sum(_count(c) for c in n.children)

print(f"Loaded corpus tree: {ROOT.name}")  # confirm the load worked and show the root's name
print(f"  {_count(ROOT)} nodes | {ROOT.count_leaves()} leaf units | "  # report total nodes and leaf count
      f"{len(ROOT.children)} top-level children")  # and how many things sit directly under the root
print("  top-level:", ", ".join(f"{c.name}({len(c.children)})" for c in ROOT.children[:12]))  # list the first dozen top-level children with their own child counts, so you can eyeball whether any are wide enough to trigger virtualisation

Loaded corpus tree: folders
  130154 nodes | 95458 leaf units | 10 top-level children
  top-level: Assay_Validation(3), CAPA(48), Change_Requests(23), IAP(23), Lab_Quality_Documents(12), Management(17), Planned_Deviations(30), Quality SOPs and Worksheets(35), Safety SOPs and Worksheets(11), Technical SOPs and Worksheets(54)


In [ ]:
# At a given node the agent can choose to either descend, backtrack, or answer (terminate traversal + retrieved chunks are sufficient), recording useful details in working memory

import json as _json  # aliased so it won't clash with any local variable named json


def _add_memory(memory: List[str], fact: str):
    # adds a fact to working memory, but trims it, skips empties and duplicates and junk like "n/a", and keeps only the most recent N so the list doesn't grow forever
    fact = _clip(fact, 500)
    if fact and fact.lower() not in ("", "none", "n/a", "(empty)") and fact not in memory:
        memory.append(fact)
        del memory[:-MEMORY_MAX_ITEMS]      # keep only the most recent N


def _parse_decision(raw: str, n_options: int) -> Dict:
    # pulls the agent's JSON decision out of the model's reply, stripping code fences and grabbing the first {...}, and if anything is malformed it falls back to a safe default rather than crashing
    s = raw.strip()
    s = re.sub(r"^```(?:json)?|```$", "", s, flags=re.M).strip()
    m = re.search(r"\{.*\}", s, flags=re.S)
    if m:
        try:
            d = _json.loads(m.group(0))
            act = str(d.get("action", "")).lower().strip()
            if act not in ("descend", "answer", "backtrack"):
                act = "descend" if n_options else "answer"
            ci = d.get("child", None)
            try:
                ci = int(ci)
            except (TypeError, ValueError):
                ci = None
            return {"action": act, "child": ci,
                    "remember": (d.get("remember") or "").strip(),
                    "reasoning": _clip(d.get("reasoning", ""), 240)}
        except Exception:
            pass
    # un-parseable -> safe default
    return {"action": "descend" if n_options else "backtrack", "child": 0 if n_options else None,
            "remember": "", "reasoning": "(unparsed model output)"}


def _ask_decision(query: str, node: "TreeNode", options: List["TreeNode"],
                  memory: List[str], can_backtrack: bool) -> Dict:
    # builds the decision prompt the agent sees at one node, the question, its working memory, the current summary, and the numbered child options, only offering backtrack when there's somewhere to back up to, then asks the model and parses the reply
    if options:
        opts_txt = "\n".join(f"[{i}] {o.name} — {_clip(o.summary, 380)}"
                             for i, o in enumerate(options))
    else:
        opts_txt = "(no remaining options here)"
    mem_txt = "\n".join(f"- {m}" for m in memory) or "(empty)"
    acts = ['"answer"']
    if options:
        acts.insert(0, '"descend"')
    if can_backtrack:
        acts.append('"backtrack"')
    prompt = (
        "You are an agent navigating a tree of document summaries to answer a "
        "question. You see only summaries (not the raw documents) and move one "
        "node at a time.\n\n"
        f"QUESTION: {query}\n\n"
        f"WORKING MEMORY (useful facts gathered so far):\n{mem_txt}\n\n"
        f"CURRENT NODE: {node.name} [{node.node_type}]\n"
        f"CURRENT SUMMARY: {_clip(node.summary, 900)}\n\n"
        f"CHILD OPTIONS:\n{opts_txt}\n\n"
        f"Choose ONE action ({', '.join(acts)}). Reply with ONLY a JSON object:\n"
        '{"reasoning": "<1-2 sentences>", '
        '"remember": "<a specific useful fact from the CURRENT summary worth keeping '
        'for the final answer, else empty>", '
        '"action": "descend|answer|backtrack", '
        '"child": <the [index] to descend into, or null>}\n'
        "Pick 'descend' with the index whose summary is most likely to lead to the "
        "answer. Pick 'answer' if working memory already answers the question or this "
        "is the most relevant place. Pick 'backtrack' if none of the options are "
        "relevant to the question."
    )
    return _parse_decision(_chat_text(prompt), len(options))


def memwalker(query: str, root: "TreeNode", strategy: str, verbose: bool = True) -> Dict:
    # the main loop, it walks the tree one node at a time using an explicit stack so it can backtrack, gathers working-memory notes and evidence as it goes, and stops when the agent says it can answer or it runs out of places to look or hits the step cap
    memory: List[str] = []
    evidence: List["TreeNode"] = []
    seen_evidence = set()
    visited = {root.node_id}
    trail: List[str] = []
    stack = [{"node": root, "options": get_nav_children(root, strategy), "tried": set()}]

    def log(msg):
        # records a step in the trail and prints it too when verbose, so you can watch the walk happen
        trail.append(msg)
        if verbose:
            print(msg)

    log(f"START at ROOT: {root.name}")
    steps = 0
    while stack and steps < MAX_STEPS:
        steps += 1
        fr = stack[-1]
        node = fr["node"]
        present = [(i, o) for i, o in enumerate(fr["options"])
                   if i not in fr["tried"] and o.node_id not in visited]
        opts = [o for _, o in present]
        dec = _ask_decision(query, node, opts, memory, can_backtrack=len(stack) > 1)
        if dec["remember"]:
            _add_memory(memory, dec["remember"])

        act = dec["action"]
        if act == "answer":
            log(f"  ANSWER here ({node.name}) — {dec['reasoning']}")
            if node.node_id not in seen_evidence:
                evidence.append(node); seen_evidence.add(node.node_id)
            break

        if act == "backtrack" or (act == "descend" and not opts):
            if len(stack) > 1:
                popped = stack.pop()
                parent = stack[-1]
                for i, o in enumerate(parent["options"]):
                    if o.node_id == popped["node"].node_id:
                        parent["tried"].add(i); break
                log(f"  BACKTRACK out of {popped['node'].name} — {dec['reasoning']}")
                continue
            log("  give up at root (nothing relevant) — answering from memory")
            break

        # descend
        ci = dec["child"]
        if ci is None or not (0 <= ci < len(opts)):
            ci = 0
        child = opts[ci]
        visited.add(child.node_id)
        child_opts = get_nav_children(child, strategy)
        if child.is_leaf() or not child_opts:
            log(f"  READ leaf: {child.name} ({child.metadata.get('source_file', '')})")
            if child.node_id not in seen_evidence:
                evidence.append(child); seen_evidence.add(child.node_id)
            if child.content:
                _add_memory(memory, _clip(child.content, 600))
            for i, o in enumerate(fr["options"]):       # don't offer it again here
                if o.node_id == child.node_id:
                    fr["tried"].add(i); break
            continue
        log(f"  DESCEND into {child.name} ({child.node_type}) — {dec['reasoning']}")
        stack.append({"node": child, "options": child_opts, "tried": set()})

    answer = _synthesize(query, memory, evidence)
    return {"answer": answer, "memory": memory, "evidence": evidence,
            "trail": trail, "steps": steps}


def _synthesize(query: str, memory: List[str], evidence: List["TreeNode"]) -> str:
    # writes the final answer, feeding the model only the working memory and the gathered evidence, asking it to stick to what was found and to cite source files, so it doesn't invent things
    mem_txt = "\n".join(f"- {m}" for m in memory) or "(none)"
    ev_parts = []
    for e in evidence[:MAX_EVIDENCE]:
        src = e.metadata.get("source_file") or e.path or e.name
        ev_parts.append(f"[{src}]\n{_clip(e.content or e.summary, 1500)}")
    ev_txt = "\n\n".join(ev_parts) or "(none)"
    prompt = (
        "Answer the question using ONLY the information gathered below. If it is "
        "insufficient, say what is missing rather than guessing.\n\n"
        f"QUESTION: {query}\n\n"
        f"WORKING MEMORY:\n{mem_txt}\n\n"
        f"GATHERED EVIDENCE (most relevant nodes):\n{ev_txt}\n\n"
        "Write a clear, specific, and thorough answer providing all relevant information. Cite source files in [brackets] where relevant."
    )
    return _chat_text(prompt)


def answer_query(query: str, strategy: str = None, verbose: bool = True) -> Dict:
    # the friendly entry point you actually call, it runs the walk, then prints a tidy report of the query, the steps taken, the working memory, and the final answer, and hands back the full result dict
    strategy = strategy or VIRTUAL_STRATEGY
    print("=" * 80)
    print(f"QUERY: {query}")
    print(f"(strategy={strategy}, max_branch={MAX_BRANCH})")
    print("-" * 80)
    res = memwalker(query, ROOT, strategy, verbose=verbose)
    print("-" * 80)
    print(f"Visited {res['steps']} step(s); kept {len(res['memory'])} memory note(s), "
          f"{len(res['evidence'])} evidence node(s).")
    print("\nWORKING MEMORY:")
    for m in res["memory"]:
        print(f"  - {m}")
    print("\nANSWER:")
    print(res["answer"])
    print("=" * 80)
    return res


print("MemWalker agent ready (working memory + backtracking). Call answer_query('...').")

MemWalker agent ready (working memory + backtracking). Call answer_query('...').


In [21]:
def compare_strategies(query: str, strategies=("none", "chunk", "cluster")):
    """side-by-side ablation."""
    out = {}
    for s in strategies:
        print("\n" + "#" * 80 + f"\n# STRATEGY = {s}\n" + "#" * 80)
        out[s] = answer_query(query, strategy=s, verbose=True)
    print("\n" + "=" * 80 + "\nSIDE-BY-SIDE ANSWERS\n" + "=" * 80)
    for s, r in out.items():
        print(f"\n[{s}] ({r['steps']} steps, {len(r['evidence'])} evidence)")
        print(textwrap.fill(r["answer"], width=92))
    return out

QUESTION = "What are the Ten Steps To CAP Laboratory Accreditation?"
_ = answer_query(QUESTION, strategy="cluster")


QUERY: What are the Ten Steps To CAP Laboratory Accreditation?
(strategy=cluster, max_branch=8)
--------------------------------------------------------------------------------
START at ROOT: folders
  DESCEND into [group 1 · 8 items] (vgroup) — The Ten Steps to CAP Laboratory Accreditation are likely documented within the Lab Quality Documents or accreditation records, which are part of group 1.
  DESCEND into Lab_Quality_Documents (folder) — The Lab Quality Documents folder likely contains accreditation toolkits and may list the Ten Steps to CAP Laboratory Accreditation.
  DESCEND into [group 2 · 7 items] (vgroup) — The Ten Steps to CAP Laboratory Accreditation are likely detailed in the accreditation toolkit documents, which are part of group 2 in the Lab Quality Documents folder.
  DESCEND into Checklists (folder) — The Ten Steps to CAP Laboratory Accreditation are likely detailed in the accreditation checklists, specifically the ACD Requirements toolkit.
  DESCEND into CAP (folder

In [22]:
_ = answer_query("What is the minimum concentration of cfDNA aliquot needed for pWGS?", strategy="cluster")

QUERY: What is the minimum concentration of cfDNA aliquot needed for pWGS?
(strategy=cluster, max_branch=8)
--------------------------------------------------------------------------------
START at ROOT: folders
  DESCEND into [group 1 · 8 items] (vgroup) — The answer likely resides in the assay validation reports for plasma‑WGS, which are under the core quality‑system collection.
  DESCEND into Assay_Validation (folder) — The assay validation documents should contain the required cfDNA concentration for plasma whole-genome sequencing (pWGS).
  DESCEND into Approved Validation Reports (folder) — The current folder overview mentions input limits but doesn't give the specific minimum cfDNA concentration; the detailed validation reports should contain that metric.
  DESCEND into [group 2 · 3 items] (vgroup) — The plasma whole-genome sequencing assay details, including input thresholds, are likely in group 2 which mentions input requirements for pWGS.
  DESCEND into Plasma Whole Genome Seq

In [23]:
_ = answer_query("What reagents are used with the Illumina NextSeq 2000?", strategy="cluster")

QUERY: What reagents are used with the Illumina NextSeq 2000?
(strategy=cluster, max_branch=8)
--------------------------------------------------------------------------------
START at ROOT: folders
  DESCEND into [group 1 · 8 items] (vgroup) — The question asks about reagents for the Illumina NextSeq 2000; the root contains groups of quality‑system documents, and group 1 likely includes assay SOPs and reagent lists relevant to sequencing instruments.
  DESCEND into Technical SOPs and Worksheets (folder) — The question asks about reagents for Illumina NextSeq 2000; sequencing protocols are likely documented in the Technical SOPs and Worksheets.
  DESCEND into [group 5 · 11 items] (vgroup) — Group 5 mentions sequencing instrument operation for Illumina NextSeq 2000, likely containing details on the reagents used with that instrument.
  DESCEND into [group 2 · 4 items] (vgroup) — The NextSeq 2000 operation SOP likely lists the specific reagents required for runs, making it the best sourc

In [24]:
_ = answer_query("What is the cleaning up process after working with blood?", strategy="cluster")

QUERY: What is the cleaning up process after working with blood?
(strategy=cluster, max_branch=8)
--------------------------------------------------------------------------------
START at ROOT: folders
  DESCEND into [group 1 · 8 items] (vgroup) — The answer likely resides in a specific SOP or biosafety worksheet within the core quality-system collection, which is in group 0.
  DESCEND into Safety SOPs and Worksheets (folder) — The cleaning up process after handling blood is likely detailed in the Safety SOPs, which cover biosafety and waste handling.
  DESCEND into [group 1 · 9 items] (vgroup) — The cleaning up process after working with blood is likely detailed in the biosafety or exposure control procedures, which are part of the core safety SOPs summarized in group 1.
  DESCEND into [group 1 · 8 items] (vgroup) — The answer likely resides in a detailed SOP within the core safety documentation, not in the eyewash plan.
  DESCEND into Exposure Control Plan.docx (document) — The clean